# RoadAnomaly — Notebook expérimental complet (livrable B)

Ce notebook couvre **tout le pipeline** demandé pour le rendu :

1. Business Understanding  
2. Data Understanding / **data mining** (inventaire, stats, visualisations)  
3. **Data cleaning** & contrôles d’intégrité  
4. **Data preparation / preprocessing** (masque binaire, resize 512×288, split, normalisation, augmentation)  
5. Modeling (E0 / E1 / E2a / E2b)  
6. Evaluation (validation + test)  
7. Deployment (Streamlit)  
8. Limites & conclusion  

**Rapport associé :** `A_Technical_Report_CRISP_DM.md`  

**Protocole figé (ne pas confondre avec d’anciennes figures letterbox 512×512) :**
- Entrée modèle : **512×288** (largeur × hauteur), **sans** letterbox  
- Split **42 / 9 / 9**, seed 42  
- Labels officiels `{0,2}` → modèle `{0,1}`  
- Modèle primaire : **E2b**, seuil opérationnel **0,55**


## 0. Setup


In [ ]:
%matplotlib inline
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Racine du dépôt (fonctionne depuis deliverable/ ou la racine)
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src").exists() else HERE.parent
assert (ROOT / "src").exists(), f"src/ introuvable depuis {HERE}"
sys.path.insert(0, str(ROOT / "src"))

DELIVERABLE = ROOT / "deliverable"
EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
print("ROOT", ROOT)
print("frames exist", (ROOT / "frames").exists())
print("manifest exist", (ROOT / "data/processed/split_manifest.csv").exists())


## 1. Business Understanding

**Problème.** Les détecteurs routiers classiques (voiture, piéton, route…) échouent sur des obstacles hors vocabulaire. RoadAnomaly (Lis et al., ICCV 2019) fournit 60 scènes annotées au pixel.

**Objectifs métier**
- Détecter l’obstacle inattendu  
- Le **localiser** (région pixel)  
- Couvrir tailles / types variés  
- Produire une sortie inspectable (image + masque)

**Important.** Lis et al. font de la **resynthèse / discordance** (souvent après entraînement sur scènes normales).  
Ce projet fait de la **segmentation sémantique binaire supervisée** sur un split 42/9/9 de RoadAnomaly. Les scores ne sont pas directement comparables au papier ICCV.


## 2. Data Understanding & data mining

Inventaire des 60 paires image/masque, statistiques d’anomalie, catégories, et contrôle des valeurs de labels.


In [ ]:
from road_anomaly import (
    binary_mask,
    discover_frames,
    load_semantic_mask,
    overlay_anomaly,
    records_to_frame,
)

records = discover_frames(ROOT)
inventory = records_to_frame(records)

label_values = sorted({v for r in records for v in r.label_values})
pixel_share = inventory["anomaly_pixels"].sum() / inventory["total_pixels"].sum()

print(f"Images appariées : {len(records)}")
print(f"Résolution : {inventory['width'].min()}×{inventory['height'].min()} (toutes identiques?)")
print(f"Résolutions distinctes : {inventory[['width','height']].drop_duplicates().shape[0]}")
print(f"Valeurs de masque trouvées : {label_values}  (attendu {{0, 2}})")
print(f"Part globale pixels anomalie : {pixel_share:.2%}")
print(f"Médiane / image : {inventory['anomaly_ratio'].median():.2%}")
print(f"Min : {inventory['anomaly_ratio'].min():.2%} ({inventory.loc[inventory['anomaly_ratio'].idxmin(),'frame_id']})")
print(f"Max : {inventory['anomaly_ratio'].max():.2%} ({inventory.loc[inventory['anomaly_ratio'].idxmax(),'frame_id']})")
print()
print("=== Data mining : répartition par catégorie ===")
print(inventory.groupby("category").agg(
    images=("frame_id", "count"),
    anomaly_ratio_mean=("anomaly_ratio", "mean"),
    anomaly_ratio_median=("anomaly_ratio", "median"),
).round(4).to_string())

# Figure mining : histogramme des ratios + counts
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].bar(inventory.groupby("category").size().index, inventory.groupby("category").size().values, color="#e85d2a")
axes[0].set_title("Images par catégorie")
axes[0].tick_params(axis="x", rotation=30)
axes[1].hist(inventory["anomaly_ratio"] * 100, bins=12, color="#243040", edgecolor="white")
axes[1].set_title("Distribution % pixels anomalie / image")
axes[1].set_xlabel("% anomalie")
fig.tight_layout()
plt.show()

print()
print("Insight : ~90% pixels fond → l'accuracy seule est trompeuse (baseline tout-fond).")
print("Métriques utiles : Precision, Recall, F1, IoU, Dice.")


### Correspondance image / masque (exemple)

Contrôle visuel : même taille H×W, labels {0,2}, overlay aligné.


In [ ]:
example_id = "animals17_E6_near_Kvaenangen"
if example_id not in set(inventory["frame_id"]):
    example_id = inventory.sort_values("anomaly_ratio").iloc[len(inventory)//2]["frame_id"]

example = inventory.loc[inventory["frame_id"] == example_id].iloc[0]
image = np.array(Image.open(example.image_path).convert("RGB"))
semantic = load_semantic_mask(Path(example.semantic_path))
mask01 = np.array(binary_mask(semantic))

print(example_id)
print("shapes image/mask", image.shape, semantic.shape)
print("valeurs officielles", np.unique(semantic).tolist())
print(f"pixels anomalie : {(semantic==2).sum()} ({example.anomaly_ratio:.1%})")

display_mask = np.stack([mask01*255, np.zeros_like(mask01), np.zeros_like(mask01)], axis=-1)
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
axes[0].imshow(image); axes[0].set_title("Photographie")
axes[1].imshow(display_mask); axes[1].set_title("Masque (rouge = anomalie)")
axes[2].imshow(overlay_anomaly(image, mask01)); axes[2].set_title("Superposition")
for ax in axes:
    ax.axis("off")
fig.tight_layout(); plt.show()


## 3. Data cleaning & contrôles d’intégrité

« Cleaning » ici = validation stricte des artefacts bruts (pas de suppression arbite du jeu) :

- chaque frame de `frame_list.json` a une image **et** un masque  
- tailles image/masque identiques  
- labels ∈ {0, 2} uniquement  
- au moins 1 pixel anomalie  
- scènes répétées (`scene_group`) ne traversent **pas** les splits


In [ ]:
issues = []
for rec in records:
    if rec.image_path is None or not Path(rec.image_path).is_file():
        issues.append((rec.frame_id, "image manquante"))
    if rec.semantic_path is None or not Path(rec.semantic_path).is_file():
        issues.append((rec.frame_id, "masque manquant"))
    bad_vals = set(rec.label_values) - {0, 2}
    if bad_vals:
        issues.append((rec.frame_id, f"labels inattendus {bad_vals}"))
    if rec.anomaly_pixels == 0:
        issues.append((rec.frame_id, "aucun pixel anomalie"))

print(f"Frames auditées : {len(records)}")
print(f"Problèmes détectés : {len(issues)}")
if issues:
    display(pd.DataFrame(issues, columns=["frame_id", "issue"]))
else:
    print("OK — inventaire propre (appariement, labels, anomalie non vide).")

# Cohérence instance vs sémantique si disponible
if "instance_agrees_with_semantic" in inventory.columns:
    disagree = int((inventory["instance_agrees_with_semantic"] == False).sum())
    print(f"Désaccords instance/sémantique : {disagree}")


## 4. Data preparation / preprocessing

### Décisions figées
| Étape | Choix | Raison |
| --- | --- | --- |
| Masque binaire | 2 → 1, 0 → 0 | Segmentation binaire |
| Resize | **512×288**, bilinéaire (RGB) / nearest (masque) | Divisible par 32 ; **pas** de letterbox |
| Split | 42/9/9 + verrou SHA | Reproductibilité ; scènes répétées en train |
| Norm E1 | `scale_0_1` | CNN scratch |
| Norm E2 | `mobilenet_v2` = `x/127.5 - 1` | MobileNetV2 Keras |
| Augmentations | **train only**, image+masque synchronisés | Pas de fuite val/test |

Code : `src/preprocessing.py`, `dataset.py`, `augmentation.py`.


In [ ]:
from preprocessing import (
    load_rgb_image,
    load_semantic_mask as load_mask_prep,
    remap_official_mask_to_binary,
    resize_image_rgb,
    resize_mask_nearest,
    normalize_image,
    preprocess_pair,
)
from dataset import load_split_manifest, verify_split_lock, RoadAnomalyDataset

manifest = load_split_manifest()
sha = verify_split_lock(manifest)
print("Split counts", manifest["split"].value_counts().to_dict())
print("split_sha", sha)
print("lock_ok", sha == EXPECTED_SPLIT)

crossing = int(manifest.groupby("scene_group")["split"].nunique().gt(1).sum())
print("scene_groups crossing splits", crossing)
assert crossing == 0
assert list(manifest.groupby("split").size().reindex(["train","val","test"])) == [42, 9, 9]

print()
print("Crosstab split × category")
print(pd.crosstab(manifest["split"], manifest["category"]).reindex(index=["train","val","test"]))

# Démo preprocessing sur une image train
demo_id = manifest.loc[manifest["split"]=="train", "frame_id"].iloc[0]
img_path = ROOT / "frames" / f"{demo_id}.webp"
msk_path = ROOT / "frames" / f"{demo_id}.labels" / "labels_semantic.png"
raw_img = load_rgb_image(img_path)
raw_msk = load_mask_prep(msk_path)
binary = remap_official_mask_to_binary(raw_msk)
packed = preprocess_pair(raw_img, raw_msk, normalize="scale_0_1", frame_id=demo_id)

print()
print(demo_id)
print("brut", raw_img.shape, "→ préparé", packed["image_uint8"].shape, packed["mask"].shape)
print("mask unique", np.unique(packed["mask"]).tolist())
print("normalize scale_0_1 range", float(packed["image"].min()), float(packed["image"].max()))
mv2 = normalize_image(packed["image_uint8"], "mobilenet_v2")
print("normalize mobilenet_v2 range", float(mv2.min()), float(mv2.max()))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].imshow(raw_img); axes[0].set_title(f"Brut {raw_img.shape[1]}×{raw_img.shape[0]}")
axes[1].imshow(packed["image_uint8"]); axes[1].set_title("Préparé 512×288")
axes[2].imshow(packed["mask"], cmap="gray", vmin=0, vmax=1); axes[2].set_title("Masque binaire {0,1}")
for ax in axes:
    ax.axis("off")
fig.suptitle("Preprocessing figé (sans letterbox)")
fig.tight_layout(); plt.show()


### Augmentations (train only) — image et masque restent alignés


In [ ]:
from augmentation import AugmentationConfig, augment_pair

cfg = AugmentationConfig.from_project_config()
base_img = packed["image_uint8"]
base_msk = packed["mask"]

fig, axes = plt.subplots(2, 4, figsize=(12.5, 5.8))
axes[0,0].imshow(base_img); axes[0,0].set_title("Original préparé")
axes[1,0].imshow(base_msk, cmap="gray", vmin=0, vmax=1); axes[1,0].set_title("Masque")
for col, seed in enumerate([1, 2, 3], start=1):
    ai, am = augment_pair(base_img, base_msk, rng=np.random.default_rng(seed), config=cfg)
    axes[0,col].imshow(ai); axes[0,col].set_title(f"Augment seed={seed}")
    axes[1,col].imshow(am, cmap="gray", vmin=0, vmax=1); axes[1,col].set_title("Masque sync")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Augmentation train-only (pas appliquée à val/test)")
fig.tight_layout(); plt.show()


## 5. Modeling

| Exp | Architecture | Points clés |
| --- | --- | --- |
| E0 | Tout fond | Baseline ; accuracy haute, Dice=0 |
| E1 | U-Net-lite scratch (`cnn.py`) | `base_filters=8`, **sans BN**, loss BCEₚₒₛ=15 + Dice |
| E2a | U-Net + MobileNetV2 | Encodeur ImageNet **gelé** |
| E2b | Idem + fine-tune | Dégel depuis `block_13_expand`, LR 1e-4 — **modèle primaire** |

Checkpointing pendant train : `val_mean_dice` @ seuil **0,5**.  
Seuil **opérationnel** choisi ensuite sur val (grille) — E2b → **0,55**.


In [ ]:
from models.baseline import AllBackgroundModel
from models.cnn import build_scratch_fcn
from models.unet_mobilenetv2 import build_unet_mobilenetv2

e0 = AllBackgroundModel()
e1 = build_scratch_fcn(288, 512, base_filters=8, dropout=0.2)
e2 = build_unet_mobilenetv2(input_height=288, input_width=512, weights=None, encoder_trainable=False)

x = np.zeros((1, 288, 512, 3), dtype=np.float32)
print("E0 proba shape", e0.predict_proba(x).shape)
print("E1", e1.name, "params", f"{e1.count_params():,}", "out", e1.predict(x, verbose=0).shape)
print("E2", e2.name, "params", f"{e2.count_params():,}", "out", e2.predict(x, verbose=0).shape)
print("Sortie : sigmoid → proba anomalie par pixel ∈ [0,1]")


## 6. Evaluation

### 6.1 Validation (sélection)
Dice moyen/image après sélection de seuil sur val. **E2b @ 0,55** est figé avant tout accès test.


In [ ]:
val_paths = [
    DELIVERABLE / "supporting_val_comparison.csv",
    ROOT / "experiments/results/phase4/phase4_E0_E1_E2_val_comparison.csv",
]
val_csv = next(p for p in val_paths if p.is_file())
val = pd.read_csv(val_csv)
cols = [c for c in ["experiment","experiment_id","threshold","mean_dice","mean_iou","mean_precision","mean_recall","mean_accuracy"] if c in val.columns]
print("Source:", val_csv)
print(val[cols].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = val["experiment"].tolist() if "experiment" in val.columns else val["experiment_id"].astype(str).tolist()
ax.bar(labels, val["mean_dice"], color=["#9aa3ad","#5c6b7a","#c45c26","#e85d2a"][:len(val)])
ax.set_ylabel("Dice mean/image (val)")
ax.set_title("Comparaison validation — sélection de modèle")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout(); plt.show()


### 6.2 Test (one-shot après gel)

Aucune retuning. Garde-fou `STAGE5_TEST_EVAL_DONE.json`.


In [ ]:
test_paths = [
    DELIVERABLE / "supporting_test_comparison.csv",
    ROOT / "experiments/results/phase4/phase4_final_test_comparison.csv",
]
test_csv = next(p for p in test_paths if p.is_file())
test = pd.read_csv(test_csv)
cols = [c for c in ["experiment_id","threshold","mean_dice","mean_iou","mean_precision","mean_recall","global_dice","global_accuracy"] if c in test.columns]
print("Source:", test_csv)
print(test[cols].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(test["experiment_id"].str.slice(0, 12), test["mean_dice"], color="#243040")
ax.set_ylabel("Dice mean/image (test)")
ax.set_title("Test one-shot — seuils figés")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout(); plt.show()

e2b_row = test.loc[test["experiment_id"].str.startswith("E2b")].iloc[0]
print(f"Primaire E2b test Dice mean/image = {e2b_row['mean_dice']:.3f} | global = {e2b_row['global_dice']:.3f}")


## 7. Deployment

Démo Streamlit (`app.py`) :

```powershell
pip install -r requirements-app.txt
streamlit run app.py
```

- Prédiction **officielle** : E2b @ **0,55**  
- Onglet **Exploration visuelle** : seuil d’affichage hors protocole (ne retune pas le modèle)  
- Guide : `docs/APP_USER_GUIDE.md`


## 8. Limites & conclusion

**Limites.** n_train=42 ; n_val=n_test=9 ; déséquilibre ; variance élevée ; segmentation parfois partielle sur gros obstacles ; lost-tires absents du test (volontairement en train).

**Conclusion.** E2b (U-Net + MobileNetV2 fine-tuné) est le meilleur modèle en validation et sur le test one-shot (Dice mean/image ≈ **0,818**). Résultat conditionné à un jeu minuscule.

**Rapport CRISP-DM :** `A_Technical_Report_CRISP_DM.md`.
